# Chargestamps — HLC vs. SLC

An SLC hit doesn't send the full waveform — only 3 fADC samples centered
on the peak plus the peak sample index. Both HLC and SLC hits carry a
chargestamp; for HLC it's redundant (the full waveform is sent too), for
SLC it's all we have.

From [1612.05093 §6.3.4](1612.05093v3.pdf):
> For in-ice DOMs, the chargestamp consists of three samples of the fADC
> waveform centered around the peak value, along with the peak sample
> number.

This notebook fetches both:
- the HLC chargestamp for the event already plotted in
  [plot_one_dom.ipynb](plot_one_dom.ipynb) (run 126491, event 30343391, DOM (83, 31))
- a representative SLC chargestamp from the same I3 file

and prints them as a table.

In [ ]:
import subprocess, textwrap, tempfile, pickle, os, sys
from pathlib import Path
import pandas as pd

ENV_SHELL = '/cvmfs/icecube.opensciencegrid.org/py3-v4.3.0/RHEL_9_x86_64/metaprojects/icetray/v1.11.1/env-shell.sh'
I3_FILE   = '/lustre/hpc/icecube/janikh/MINIONS_DA_sample_2015_v5.i3.zst'

def run_in_icetray(python_code, timeout=3600):
    tmp_py  = Path(tempfile.mkstemp(suffix='.py')[1])
    tmp_pkl = Path(tempfile.mkstemp(suffix='.pkl')[1])
    tmp_py.write_text(textwrap.dedent(python_code))
    proc = subprocess.Popen(
        [ENV_SHELL, 'python', '-u', str(tmp_py)],
        env={**os.environ, 'OUT_PICKLE': str(tmp_pkl), 'PYTHONUNBUFFERED': '1'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line.rstrip()); sys.stdout.flush()
    proc.wait(timeout=timeout)
    if proc.returncode != 0:
        raise RuntimeError(f'icetray subprocess failed (rc={proc.returncode})')
    return tmp_pkl

## Fetch chargestamps

Scans the I3 file until the HLC target event is found. The SLC sample is
picked up trivially along the way (any non-LC launch with a clear peak).

In [ ]:
HLC_TARGET_RUN   = 126491
HLC_TARGET_EVENT = 30343391
HLC_TARGET_OM    = (83, 31, 0)

SLC_MIN_PEAK_ABOVE_BASELINE = 30   # ADC counts above baseline
SLC_SKIP                    = 17   # pick the (N+1)-th matching SLC — bump for a different one

MAX_FRAMES_TO_SCAN          = 200000

GCD_FILE = '/lustre/hpc/icecube/janikh/GeoCalibDetectorStatus_2015.57161_V0.i3.gz'

fetch_code = f'''
    import os, math, pickle, time
    from icecube import icetray, dataio, dataclasses
    from icecube.icetray import I3Units
    HLC_RUN, HLC_EV, HLC_OM = {HLC_TARGET_RUN}, {HLC_TARGET_EVENT}, {HLC_TARGET_OM!r}

    f = dataio.I3File("{GCD_FILE}")
    cal_obj, det_obj = None, None
    while f.more():
        fr = f.pop_frame()
        if "I3Calibration" in fr: cal_obj = fr["I3Calibration"]
        if "I3DetectorStatus" in fr: det_obj = fr["I3DetectorStatus"]
        if cal_obj and det_obj: break

    FE_R, E_CHG = 50.0, 1.602176634e-19
    def cal_consts(om):
        dc = cal_obj.dom_cal[om]
        ds = det_obj.dom_status[om]
        hv = float(ds.pmt_hv) / I3Units.V
        g  = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
        fadc_gain_V_per_count = float(dc.fadc_gain) / I3Units.V
        baseline_counts = float(dc.fadc_beacon_baseline)
        kf = 1.0 / (FE_R * g * E_CHG)
        return fadc_gain_V_per_count, baseline_counts, kf

    f = dataio.I3File("{I3_FILE}")
    hlc, slc = None, None
    slc_skipped = 0
    seen = 0
    t0 = time.time()
    while f.more() and seen < {MAX_FRAMES_TO_SCAN} and hlc is None:
        fr = f.pop_frame()
        if "InIceRawData" not in fr: continue
        seen += 1
        if seen % 10000 == 0:
            print(f"  scanned {{seen}} frames ({{time.time()-t0:.0f}}s)  "
                  f"hlc={{hlc is not None}} slc={{slc is not None}}", flush=True)
        rd = fr["InIceRawData"]
        hdr = fr["I3EventHeader"]
        is_hlc_event = (hdr.run_id == HLC_RUN and hdr.event_id == HLC_EV)
        for om, launches in rd:
            om_t = (int(om.string), int(om.om), int(om.pmt))
            for L in launches:
                cs = list(L.raw_charge_stamp)
                if len(cs) != 3: continue

                if hlc is None and is_hlc_event and om_t == HLC_OM and L.lc_bit:
                    fg, bl, kf = cal_consts(om)
                    hlc = {{
                        "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                        "om": om_t,
                        "launch_time_ns": float(L.time),
                        "lc_bit": True,
                        "raw_charge_stamp": cs,
                        "charge_stamp_highest_sample": int(L.charge_stamp_highest_sample),
                        "fadc_gain_V_per_count": fg,
                        "fadc_baseline_counts": bl,
                        "pe_per_voltsecond": kf,
                    }}

                if slc is None and (not L.lc_bit):
                    if max(cs) - min(cs) >= {SLC_MIN_PEAK_ABOVE_BASELINE}:
                        if slc_skipped < {SLC_SKIP}:
                            slc_skipped += 1
                            continue
                        fg, bl, kf = cal_consts(om)
                        slc = {{
                            "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                            "om": om_t,
                            "launch_time_ns": float(L.time),
                            "lc_bit": False,
                            "raw_charge_stamp": cs,
                            "charge_stamp_highest_sample": int(L.charge_stamp_highest_sample),
                            "fadc_gain_V_per_count": fg,
                            "fadc_baseline_counts": bl,
                            "pe_per_voltsecond": kf,
                        }}

    out = {{"hlc": hlc, "slc": slc, "frames_scanned": seen}}
    with open(os.environ["OUT_PICKLE"], "wb") as g:
        pickle.dump(out, g, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"done — scanned {{seen}} frames")
    print(f"  HLC found: {{hlc is not None}}")
    print(f"  SLC found: {{slc is not None}}  (skipped {{slc_skipped}} earlier matches)")
'''

pkl_path = run_in_icetray(fetch_code)
with open(pkl_path, 'rb') as f:
    out = pickle.load(f)
hlc = out['hlc']
slc = out['slc']
assert hlc is not None and slc is not None, 'one of the hits was not found'
print(f"SLC chosen: run {slc['event']['run_id']} event {slc['event']['event_id']} "
      f"DOM {slc['om']}  cs = {slc['raw_charge_stamp']}")

## Table

In [ ]:
# Raw I3 fields only — exactly what is stored on the I3DOMLaunch.
# No derived per-sample times, no PE conversion.
def row(label, hit):
    cs = hit['raw_charge_stamp']
    return {
        'type'              : label,
        'run'               : hit['event']['run_id'],
        'event'             : hit['event']['event_id'],
        'DOM (string, om)'  : f"({hit['om'][0]}, {hit['om'][1]})",
        'lc_bit'            : hit['lc_bit'],
        'L.time [ns]'       : f"{hit['launch_time_ns']:.1f}",
        'raw_charge_stamp'  : cs,
        'peak idx'          : hit['charge_stamp_highest_sample'],
    }

df = pd.DataFrame([row('HLC', hlc), row('SLC', slc)])
df

In [ ]:
# Save raw table as text. All values are straight from
# the I3DOMLaunch - no derived quantities.
PLOTS_DIR = Path('plots'); PLOTS_DIR.mkdir(exist_ok=True)
txt_path = PLOTS_DIR / 'chargestamp_comparison.txt'
txt_path.write_text(df.to_string(index=False) + '\n')
print(f'saved {txt_path}')
print()
print(df.to_string(index=False))